# Lab 02: LSTM and GRU Comparison

**Module 02** | Read `notes/02-lstm-gru.pdf` before running this notebook.

Train identical LSTM and GRU classifiers on the same data and compare final loss, wall-clock time, and parameter count.

Run every cell top to bottom. Code is complete, no fill-in sections. Markdown cells explain what each block does and why.


## Runtime device

PyTorch can run on your NVIDIA GPU through CUDA or fall back to CPU. GPU execution moves matrix operations off the host and typically speeds up neural network training by a large factor. The next cell detects what is available and prints the active device so you know whether to expect fast training or should keep batch sizes small.


In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"CUDA enabled, {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {props.total_memory / 1e9:.1f} GB")
else:
    print("CUDA not available, using CPU. Labs still run; training may take longer.")


Vanilla RNNs struggle with long-range dependencies because gradients shrink through many timesteps. LSTM and GRU cells add gating mechanisms that control what to remember and forget, which usually stabilizes training on sequential data.

This lab keeps the same character-level next-token task as Lab 01 so differences in final loss, wall-clock time, and parameter count reflect the cell architecture, not a change in dataset or objective.


We rebuild the corpus pipeline identically to Lab 01. Shared data preparation is deliberate: any metric gap between LSTM and GRU is attributable to the recurrent cell, not preprocessing.


In [ ]:
import time
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

ROOT = Path("..").resolve()
SEQ_LEN = 32
BATCH_SIZE = 64
HIDDEN_SIZE = 128
EPOCHS = 5
LR = 1e-2

CORPUS = "sequence modeling practice text " * 50
chars = sorted(set(CORPUS))
vocab_size = len(chars)
char_to_idx = {ch: i for i, ch in enumerate(chars)}

indices = torch.tensor([char_to_idx[c] for c in CORPUS], dtype=torch.long)
inputs, targets = [], []
for start in range(len(indices) - SEQ_LEN):
    inputs.append(indices[start : start + SEQ_LEN])
    targets.append(indices[start + SEQ_LEN])
inputs = torch.stack(inputs)
targets = torch.stack(targets)

dataset = TensorDataset(inputs, targets)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)
print(f"Examples: {len(dataset):,} | Vocab: {vocab_size}")


Both models share the same outer structure: embedding, recurrent core, and linear classifier. The only swap is `nn.LSTM` versus `nn.GRU`. Parameter counts will differ slightly because LSTM maintains separate cell and hidden states.

A small factory function trains either model with the same optimizer and epoch count, which keeps the comparison fair.


In [ ]:
class CharRecurrentClassifier(nn.Module):
    def __init__(self, vocab_size: int, hidden_size: int, cell: str = "lstm"):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, hidden_size)
        if cell == "lstm":
            self.rnn = nn.LSTM(hidden_size, hidden_size, batch_first=True)
        elif cell == "gru":
            self.rnn = nn.GRU(hidden_size, hidden_size, batch_first=True)
        else:
            raise ValueError(f"Unknown cell: {cell}")
        self.fc = nn.Linear(hidden_size, vocab_size)
        self.cell = cell

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        embedded = self.embed(x)
        output, _ = self.rnn(embedded)
        return self.fc(output[:, -1, :])


def train_model(model: nn.Module) -> float:
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    final_loss = 0.0

    for epoch in range(1, EPOCHS + 1):
        model.train()
        epoch_loss = 0.0
        for batch_x, batch_y in loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)
            optimizer.zero_grad()
            logits = model(batch_x)
            loss = criterion(logits, batch_y)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * batch_x.size(0)
        final_loss = epoch_loss / len(dataset)
        print(f"  Epoch {epoch:02d}, loss {final_loss:.4f}")
    return final_loss


We train LSTM and GRU sequentially and record wall-clock time plus parameter counts. On CUDA you should see shorter times for the same epoch count; the relative ranking between cells is usually more informative than absolute seconds.


In [ ]:
results = []
for name, cell in [("LSTM", "lstm"), ("GRU", "gru")]:
    print(f"\nTraining {name}...")
    model = CharRecurrentClassifier(vocab_size, HIDDEN_SIZE, cell=cell)
    params = sum(p.numel() for p in model.parameters())

    torch.manual_seed(42)
    if device.type == "cuda":
        torch.cuda.manual_seed_all(42)

    start = time.perf_counter()
    final_loss = train_model(model)
    elapsed = time.perf_counter() - start
    results.append((name, final_loss, elapsed, params))

print("\nComparison")
print(f"{'Model':<8} {'Final loss':>12} {'Time (s)':>10} {'Params':>10}")
print("-" * 44)
for name, loss, elapsed, params in results:
    print(f"{name:<8} {loss:>12.4f} {elapsed:>10.2f} {params:>10,}")
